# Lexicon only — Devanagari spellings of English words (for the script-shift condition)

Runs ONLY the word-list stage of `step2_summaries_eval.ipynb`, with identical code, settings and prompt, so every
answer saved here is found and reused by the main notebook (0 repeat calls).

- No GPU needed (CPU runtime is fine).
- Needs the team folder `hinglish_asr_sum` in My Drive (or its shortcut) and `GEMINI_API_KEY` in Colab Secrets.
- Keep `STEP2_N_WINDOWS` the same as in the main notebook (16) — it decides which English words are asked for.
- Resumable: re-running skips finished batches. Output: `outputs/step2/script_shift_lexicon.csv`.

In [ ]:
# ===================== CONFIG (edit here only) =====================
SEED = 13
ASR_TAG = "vad15"                                  # which Step-1 ASR outputs to use
SENSITIVITY_EXCLUDE = ["W07", "W24"]               # windows with VAD/truncation deletions (Step 1 v2)
# Number of windows evaluated in Step 2: the first N window ids (fixed by Step 1's seeded selection, chosen before any
# summary was scored). None = all. Reduced from 25 to 16 because of the Gemini free-tier quota.
STEP2_N_WINDOWS = 16

USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/hinglish_asr_sum"
LOCAL_ROOT = "/content/hinglish_asr_sum"

# --- Gemini: reference writer AND fact judge (one model for both, never the summariser) ---
GEMINI_MODEL = "gemini-3.8-flash"     # pick a model that has free-tier quota in AI Studio > Rate limits
GEMINI_THINKING_LEVEL = None          # None = model default; e.g. "low" if supported by the model
MIN_SECONDS_BETWEEN_CALLS = 6.5       # ~9 requests/min; raise if you see many 429s
MAX_RETRIES = 8                       # per call, exponential backoff on 429 / 5xx
MAX_INVALID_JSON_RETRIES = 2
GEMINI_TIMEOUT_SEC = 180              # hard limit per request; a timeout is retried like a 5xx

# --- Summariser ---
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
QWEN_BATCH = 4
QWEN_MAX_NEW_TOKENS = 320

# --- Conditions ---
NOISE_LANGS = ["hi", "en"]
NOISE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5]          # fraction of that language's tokens corrupted
NOISE_SEED = 0
NOISE_PROFILE_MODEL = "trelis-hinglish-preview"   # S/D/I mix taken from this model's real errors
HINGLISH_OUTPUT_CONDITIONS = ["oracle", "asr:whisper-small", "asr:trelis-hinglish-preview"]
# Script-shift condition (English words rewritten in Devanagari) needs a Gemini-made word list. Optional extra,
# not part of the core design; False skips the lexicon stage and the condition entirely.
RUN_SCRIPT_SHIFT = False

# --- Fact tagging from the evidence span's tokens (share of Devanagari among Hindi+English tokens) ---
TAG_HI_MIN = 0.6
TAG_EN_MAX = 0.4
EVIDENCE_MIN_MATCH = 0.90             # fuzzy match ratio for an evidence excerpt to count as located

# --- Evaluation ---
N_BOOT = 2000
SPOTCHECK_N_REF = 10
SPOTCHECK_N_JUDGE = 30
STABILITY_N = 20                      # per_summary mode: summaries re-judged to measure judge self-agreement
# "per_window": ONE Gemini call judges every summary of a window (shuffled, blind labels S1..Sk) against that
# window's facts -> ~1 call per window instead of ~17. "per_summary": one call per summary (original design).
JUDGE_MODE = "per_window"
STABILITY_WINDOWS = 3                 # per_window mode: windows judged a second time in a different random order

In [ ]:
RUN_SCRIPT_SHIFT = True        # this notebook exists only to build the word list

In [ ]:
# ===================== INSTALL + IMPORTS =====================
import subprocess, sys, importlib
PINS = ["httpx", "jiwer==4.0.0", "indic_transliteration==2.3.82"]   # lexicon-only: no GPU packages
# Keep Colab's own GPU build of PyTorch: pin whatever torch/torchvision/torchaudio are already installed, so pip can
# never replace them with a CPU-only build while installing bitsandbytes / bert-score / accelerate.
import importlib.metadata as _md
KEEP_TORCH = []
for _p in ("torch", "torchvision", "torchaudio"):
    try:
        KEEP_TORCH.append(f"{_p}=={_md.version(_p)}")
    except _md.PackageNotFoundError:
        pass
print("keeping:", KEEP_TORCH)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS, *KEEP_TORCH], check=True)
importlib.invalidate_caches()

import os, re, gc, json, glob, time, math, random, shutil, hashlib, platform, unicodedata, difflib
from datetime import datetime, timezone
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from pydantic import BaseModel

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", gpu or "NONE — Qwen and BERTScore need the T4")
try:
    import torch
    print(f"torch {torch.__version__} | built for CUDA {torch.version.cuda} | GPU usable: {torch.cuda.is_available()}")
    if gpu and not torch.cuda.is_available():
        print("WARNING: a GPU is attached but this torch cannot use it. Runtime > Disconnect and delete runtime, "
              "then Run all again.")
except ImportError:
    print("torch not installed")

In [ ]:
# ===================== PATHS, LOGGING, CACHE, STEP-1 INPUTS =====================
IN_COLAB = "google.colab" in sys.modules
ROOT = LOCAL_ROOT
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = DRIVE_ROOT
OUT_DIR, LOG_DIR = f"{ROOT}/outputs", f"{ROOT}/logs"
S2_DIR, CACHE_DIR, FIG_DIR = f"{ROOT}/outputs/step2", f"{ROOT}/cache", f"{ROOT}/outputs/step2/figures"
for d in (S2_DIR, f"{CACHE_DIR}/gemini", f"{CACHE_DIR}/qwen", FIG_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

PROMPT_LOG = f"{LOG_DIR}/prompts_log.jsonl"
RUN_CONFIG = f"{LOG_DIR}/run_config_lexicon_only.json"

def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def log_prompt(step, component, model, prompt, params):
    rec = dict(ts=now_utc(), step=step, component=component, model=model, prompt=prompt, params=params)
    with open(PROMPT_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def update_run_config(**kv):
    cfg = json.load(open(RUN_CONFIG)) if os.path.exists(RUN_CONFIG) else {}
    cfg.update(kv)
    json.dump(cfg, open(RUN_CONFIG, "w"), indent=2, ensure_ascii=False, default=str)

def pkg_version(name):
    try:
        import importlib.metadata as md; return md.version(name)
    except Exception:
        return None

def sha(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

sel = pd.read_csv(f"{OUT_DIR}/selected_windows.csv", keep_default_na=False)
hyps = pd.read_csv(f"{OUT_DIR}/asr_hyps_{ASR_TAG}.csv", keep_default_na=False)
hyps = hyps.drop_duplicates(["window_id", "model"], keep="last")
WINDOWS = sorted(sel.window_id)
ASR_MODELS_FOUND = sorted(hyps.model.unique())
cnt = hyps.groupby("model").window_id.nunique()
assert (cnt == len(WINDOWS)).all(), f"incomplete ASR outputs: {cnt.to_dict()}"
sel_all, hyps_all, ALL_WINDOWS = sel.copy(), hyps.copy(), list(WINDOWS)   # all windows: used for noise patterns
if STEP2_N_WINDOWS:
    WINDOWS = ALL_WINDOWS[:STEP2_N_WINDOWS]
    sel = sel[sel.window_id.isin(WINDOWS)].reset_index(drop=True)
    hyps = hyps[hyps.window_id.isin(WINDOWS)].reset_index(drop=True)
print(f"Step 2 windows: {len(WINDOWS)} ({WINDOWS[0]}–{WINDOWS[-1]}) of {len(ALL_WINDOWS)} | ASR models: {ASR_MODELS_FOUND}")

update_run_config(started=now_utc(), python=platform.python_version(), gpu=gpu, seed=SEED,
                  config={k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict, type(None)))},
                  packages={p: pkg_version(p) for p in ["google-genai", "transformers", "bitsandbytes", "accelerate",
                                                        "torch", "jiwer", "rouge-score", "bert-score", "pydantic"]})

In [ ]:
# ===================== TEXT UTILITIES (identical to Step 1) =====================
import jiwer
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

DEVA_RE = re.compile(r"[ऀ-ॿ]")
LATIN_RE = re.compile(r"[a-z]")
DEVA_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
DEVA_CONSONANTS = set(chr(c) for c in list(range(0x0915, 0x093A)) + list(range(0x0958, 0x0960)))
NUKTA = "़"

def norm_text(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"<[^>]*>|\[[^\]]*\]", " ", s).lower().translate(DEVA_DIGITS)
    s = "".join(ch if unicodedata.category(ch)[0] in "LMN" else " " for ch in s)
    return " ".join(s.split())

def token_script(tok):
    if DEVA_RE.search(tok):
        return "hi"
    if LATIN_RE.search(tok):
        return "en"
    return "other"

def romanize_token(tok):
    if not DEVA_RE.search(tok):
        return tok
    t = tok.replace(NUKTA, "")
    r = transliterate(t, sanscript.DEVANAGARI, sanscript.IAST)
    if len(t) > 1 and t[-1] in DEVA_CONSONANTS and r.endswith("a"):
        r = r[:-1]
    for a, b in (("ṃ", "n"), ("ṁ", "n"), ("~", "n"), ("ś", "sh"), ("ṣ", "sh"), ("ṛ", "ri"), ("ḥ", "h")):
        r = r.replace(a, b)
    r = re.sub(r"c(?!h)", "ch", r)
    r = "".join(ch for ch in unicodedata.normalize("NFD", r) if not unicodedata.combining(ch)).lower()
    r = re.sub(r"[^a-z0-9]", "", r)
    return r or tok

def romanize_text(normed):
    return " ".join(romanize_token(t) for t in normed.split())

def align_stats(ref, hyp):
    """WER and per-language error rates of hyp against ref (both normalised)."""
    rt = ref.split(); scripts = [token_script(t) for t in rt]
    tot = Counter(scripts)
    if not hyp.strip():
        err, S, D, I = Counter(scripts), 0, len(rt), 0
    else:
        o = jiwer.process_words(ref, hyp)
        err = Counter()
        for ch in o.alignments[0]:
            if ch.type != "equal":
                for i in range(ch.ref_start_idx, ch.ref_end_idx):
                    err[scripts[i]] += 1
        S, D, I = o.substitutions, o.deletions, o.insertions
    return dict(wer=(S + D + I) / len(rt), hi_err=err["hi"] / tot["hi"] if tot["hi"] else np.nan,
                en_err=err["en"] / tot["en"] if tot["en"] else np.nan, S=S, D=D, I=I)

assert romanize_text("क्या है निष्पादित statement") == "kya hai nishpadit statement"
sel["ref_norm"] = sel.ref_norm.map(norm_text)          # idempotent
sel_all["ref_norm"] = sel_all.ref_norm.map(norm_text)

In [ ]:
# ===================== GEMINI CLIENT: cache, pacing, retries, prompt log =====================
import httpx

# Direct REST call to the Interactions endpoint (https://ai.google.dev/gemini-api/docs/api-key shows this request).
# Reasons for not using the google-genai SDK here: AI Studio now issues "AQ." authentication keys, which some SDK
# paths reject (401 ACCESS_TOKEN_TYPE_UNSUPPORTED); and a plain HTTP call shows the server's exact error text.
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/interactions"
MAX_CONTINUATION_HOPS = 8

def _get_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("GEMINI_API_KEY")
            if k:
                return k.strip().strip('"').strip("'")
        except Exception:
            pass
    k = os.environ.get("GEMINI_API_KEY")
    return k.strip().strip('"').strip("'") if k else k

class GeminiHTTPError(Exception):
    def __init__(self, status_code, body):
        self.status_code = status_code
        super().__init__(f"HTTP {status_code}: {str(body)[:1500]}")

_http = httpx.Client(timeout=GEMINI_TIMEOUT_SEC)

def _post(body):
    key = _get_key()
    if not key:
        raise GeminiStop("No GEMINI_API_KEY in Colab Secrets (or the environment) — Gemini stages skipped.")
    r = _http.post(GEMINI_URL, headers={"x-goog-api-key": key, "Content-Type": "application/json"}, json=body)
    if r.status_code != 200:
        raise GeminiHTTPError(r.status_code, r.text)
    return r.json()

def _steps_text(steps):
    """Text of the final model output (same rule as the SDK's output_text: skip thoughts, stop at user input)."""
    parts, collecting = [], False
    for step in reversed(steps):
        t = step.get("type")
        if t == "user_input":
            break
        if t == "thought":
            continue
        if t != "model_output":
            if collecting:
                break
            continue
        stop = False
        for item in reversed(step.get("content") or []):
            if item.get("type") != "text":
                if collecting:
                    stop = True; break
                continue
            collecting = True
            parts.append(item.get("text") or "")
        if stop:
            break
    return "".join(reversed(parts))

def _api_create(req):
    """The only network call to Gemini. store=False (nothing kept server-side); if the server stops at its
    per-request decode limit (status requires_continuation), resend with the continuation token."""
    from types import SimpleNamespace
    body = {**req, "store": False}
    steps, d = [], {}
    for hop in range(1, MAX_CONTINUATION_HOPS + 1):
        d = _post(body)
        hs = d.get("steps") or []
        if hop > 1:
            while hs and hs[0].get("type") == "user_input":
                hs = hs[1:]
        steps += hs
        if d.get("status") == "requires_continuation" and d.get("continuation_token"):
            body = {**req, "store": False, "continuation_token": d["continuation_token"]}
            continue
        break
    if d.get("status") == "failed":
        raise GeminiHTTPError(503, f"interaction failed: {json.dumps(d.get('errors'))[:800]}")
    return SimpleNamespace(output_text=_steps_text(steps), model=d.get("model"), id=d.get("id"),
                           usage=d.get("usage"), status=d.get("status"), hops=hop)

def _response_format(schema):
    return {"type": "text", "mime_type": "application/json", "schema": schema.model_json_schema()}

def _model_version(m):
    g = re.search(r"gemini-(\d+(?:\.\d+)?)", m)
    return float(g.group(1)) if g else None

# Gemini 3.6+ and 4 ignore temperature (per the API docs, Oct 2026); send temperature=0 only to older models.
_v = _model_version(GEMINI_MODEL)
SEND_TEMPERATURE = _v is not None and _v < 3.6
print(f"Gemini model: {GEMINI_MODEL} | temperature=0 sent: {SEND_TEMPERATURE}"
      + ("" if SEND_TEMPERATURE else " (model ignores temperature; reproducibility comes from the response cache)"))

class GeminiStop(Exception):
    """Raised when retries are exhausted (usually the daily free-tier quota)."""

class GeminiAuthError(Exception):
    """Raised on 401/403: the key is missing, wrong or not a Gemini API key. Not retried."""

def check_gemini_key():
    """Shape check without revealing the key; then one tiny call so a bad key fails here, not mid-stage."""
    k = _get_key()
    if not k:
        print("WARNING: no GEMINI_API_KEY — Gemini stages will stop immediately; non-Gemini stages (Qwen) still run.")
        return
    note = " | PROBLEM: looks like an OAuth access token, not an API key" if k.startswith("ya29.") else ""
    print(f"key: {len(k)} chars, starts '{k[:3]}…'{note}")
    class Ping(BaseModel):
        ok: bool
    try:
        r = _api_create(dict(model=GEMINI_MODEL, input='Return {"ok": true}.',
                             generation_config={"max_output_tokens": 1024},
                             response_format=_response_format(Ping), response_mime_type="application/json"))
        Ping.model_validate_json(r.output_text)
        print(f"Gemini key works with {GEMINI_MODEL} (response model: {r.model}, status: {r.status})")
    except Exception as e:
        code = getattr(e, "status_code", None)
        if code == 429 or isinstance(e, GeminiStop):
            print("WARNING: Gemini quota exhausted (429) — Gemini stages will stop quickly; non-Gemini stages "
                  "(conditions, Qwen summaries) still run. Re-run after the quota resets (midnight Pacific).")
            return
        hint = {401: "key rejected: create a new key at https://aistudio.google.com/apikey and update the Secret",
                403: "key not allowed to use this API/model: check the key's project in AI Studio",
                404: "model not available to this key: change GEMINI_MODEL",
                400: "request rejected: send me this message",
                429: "rate limit / quota: wait and re-run"}.get(code, "send me this message")
        raise GeminiAuthError(f"Gemini test call failed ({type(e).__name__}): {str(e)[:1200]}\n=> {hint}") from None

_last_call = [0.0]
def gemini_json(template_id, system, user, schema, max_output_tokens, variant=0, meta=None):
    """Structured JSON call with disk cache. Returns (parsed pydantic object, cache record)."""
    gc_ = {"max_output_tokens": max_output_tokens}
    if SEND_TEMPERATURE:
        gc_["temperature"] = 0.0
    if GEMINI_THINKING_LEVEL:
        gc_["thinking_level"] = GEMINI_THINKING_LEVEL
    req = dict(model=GEMINI_MODEL, system_instruction=system, input=user, generation_config=gc_,
               response_format=_response_format(schema), response_mime_type="application/json")
    key = sha(dict(req=req, variant=variant))
    path = f"{CACHE_DIR}/gemini/{key}.json"
    # Reuse a response saved by the earlier notebook version (same model, prompts and schema; its request lacked
    # the two format fields added later, so its cache key differs). Copied under the new key, marked legacy.
    legacy_req = {k: v for k, v in req.items() if k != "response_mime_type"}
    legacy_req["response_format"] = {k: v for k, v in req["response_format"].items() if k != "type"}
    legacy_path = f"{CACHE_DIR}/gemini/{sha(dict(req=legacy_req, variant=variant))}.json"
    if not os.path.exists(path) and os.path.exists(legacy_path):
        try:
            old = json.load(open(legacy_path, encoding="utf-8"))
            schema.model_validate_json(old["output_text"])
            old.update(legacy_request_format=True, legacy_cache_file=os.path.basename(legacy_path))
            json.dump(old, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
            print(f"   reused response from the earlier notebook version ({template_id}, {meta})", flush=True)
        except Exception as e:
            print(f"   could not reuse earlier response ({type(e).__name__}); calling Gemini", flush=True)
    if os.path.exists(path):
        rec = json.load(open(path, encoding="utf-8"))
        log_prompt("2", template_id, GEMINI_MODEL, None,
                   dict(cache_key=key, cache_hit=True, variant=variant, meta=meta))
        return schema.model_validate_json(rec["output_text"]), rec
    bad_json, attempt = 0, 0
    while True:
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        t_call = time.time()
        if template_id != "judge":
            print(f"   calling Gemini ({template_id}, {meta}) ...", flush=True)
        try:
            r = _api_create(req)
        except Exception as e:
            # The Interactions API raises its own classes (status in .status_code); older paths use .code.
            code = getattr(e, "status_code", None) or getattr(e, "code", None)
            transient = code in (429, 500, 502, 503, 504) or (code is None and type(e).__name__ in (
                "APIConnectionError", "APITimeoutError", "ConnectError", "ReadTimeout", "RemoteProtocolError",
                "TimeoutException", "WriteTimeout", "PoolTimeout", "ConnectTimeout"))
            if code in (401, 403):
                raise GeminiAuthError(
                    f"Gemini rejected the API key ({code}). Create a key at https://aistudio.google.com/apikey, "
                    "store it in Colab Secrets as GEMINI_API_KEY with notebook access ON, then re-run.") from e
            if transient and attempt < MAX_RETRIES:
                attempt += 1
                back = min(300, 10 * 2 ** (attempt - 1)) + random.Random(key + str(attempt)).uniform(0, 3)
                print(f"   Gemini {code or type(e).__name__}; retry {attempt}/{MAX_RETRIES} in {back:.0f}s")
                time.sleep(back)
                continue
            if transient:
                raise GeminiStop(f"Gemini {code or type(e).__name__} after {MAX_RETRIES} retries (usually the "
                                 "daily quota) — run again later; finished calls are cached.") from e
            raise
        text = getattr(r, "output_text", None) or ""
        try:
            parsed = schema.model_validate_json(text)
        except Exception:
            bad_json += 1
            if bad_json > MAX_INVALID_JSON_RETRIES:
                raise
            print(f"   invalid JSON from Gemini ({template_id}); retry {bad_json}")
            continue
        if template_id != "judge":
            print(f"   ... done in {time.time() - t_call:.0f}s", flush=True)
        usage = getattr(r, "usage", None)
        rec = dict(created=now_utc(), template_id=template_id, variant=variant, request=req, output_text=text,
                   response_model=getattr(r, "model", None), response_id=getattr(r, "id", None),
                   status=getattr(r, "status", None), hops=getattr(r, "hops", None),
                   usage=usage.model_dump() if hasattr(usage, "model_dump") else usage, meta=meta)
        json.dump(rec, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        log_prompt("2", template_id, GEMINI_MODEL, dict(system=system, input=user),
                   dict(cache_key=key, cache_hit=False, variant=variant, generation_config=gc_,
                        response_model=rec["response_model"], meta=meta))
        return parsed, rec

check_gemini_key()   # fails fast with a clear message if the key or model is wrong

In [ ]:
# ===================== PROMPT TEMPLATES (logged once per run) =====================
class Fact(BaseModel):
    fact: str
    evidence: str

class Reference(BaseModel):
    summary: str
    facts: list[Fact]

class Pair(BaseModel):
    word: str
    devanagari: str

class Lexicon(BaseModel):
    items: list[Pair]

class Judgement(BaseModel):
    id: int
    supported: bool

class Judgements(BaseModel):
    judgements: list[Judgement]

SYSTEM_REF = ("You are a careful annotator building evaluation data for summarisation research. "
              "Follow the instructions exactly and output only JSON that matches the schema.")
USER_REF = """Below is the exact human transcript of a segment of a recorded software tutorial. The speaker code-mixes Hindi and English: Hindi is mostly written in Devanagari and English mostly in Latin script, but some English words are written in Devanagari.

Produce two things.

1. "summary": an English summary of the segment's content, 120-180 words, plain prose, covering the main steps, concepts and facts taught. Do not mention the speaker, the tutorial series, greetings, credits, funding or websites.

2. "facts": 8 to 15 atomic facts that a good summary of this segment should contain.
- Each fact is one short, self-contained English sentence stating a single piece of content (a step, setting, definition, command or result).
- Exclude greetings, credits, acknowledgements, funding and website information.
- Facts must not overlap.
- For each fact, "evidence" must be copied EXACTLY, character for character, from the transcript: one contiguous excerpt of 3 to 25 words that states the fact. Do not translate, transliterate, correct or shorten words inside the excerpt.

Transcript:
<<<
{transcript}
>>>"""

SYSTEM_LEX = "You are a precise Hindi-English transliteration assistant. Output only JSON that matches the schema."
USER_LEX = """For each English word below, give the spelling a Hindi speaker would normally use when writing that English word in Devanagari script. Transliterate the pronunciation; do not translate the meaning. Examples: click -> क्लिक, file -> फ़ाइल, terminal -> टर्मिनल. For abbreviations, spell out the letter names (pdf -> पीडीएफ). Return exactly one item per input word, with the input word copied unchanged into "word".

Words:
{words}"""

SYSTEM_JUDGE = "You are a strict, consistent evaluator of summary content. Output only JSON that matches the schema."
USER_JUDGE = """Decide, for each numbered fact, whether the SUMMARY conveys it.

A fact is supported if the summary states its core information or clearly implies it, even in different words, in another language (English, Hindi or Hinglish) or in another script. It is not supported if it is missing, only vaguely related, or contradicted. Judge each fact independently, using only the summary.

SUMMARY:
<<<
{summary}
>>>

FACTS:
{facts}

Return exactly one judgement per fact id."""

USER_JUDGE_BATCH = """You will see several SUMMARIES of the same lecture segment, labelled S1, S2, ..., and one numbered list of FACTS.

For every summary and every fact, decide whether that summary conveys the fact. A fact is supported if the summary states its core information or clearly implies it, even in different words, in another language (English, Hindi or Hinglish) or in another script. It is not supported if it is missing, only vaguely related, or contradicted. Judge each summary on its own, using only that summary's text; do not compare summaries with each other.

FACTS:
{facts}

SUMMARIES:
{summaries}

Return one entry per summary label, each with exactly one judgement per fact id."""

class SummaryJudgements(BaseModel):
    summary_id: str
    judgements: list[Judgement]

class BatchJudgements(BaseModel):
    results: list[SummaryJudgements]

SYSTEM_SUM = "You are a helpful assistant that summarises lecture transcripts."
USER_SUM_EN = """The following is an automatic transcript of part of a recorded computer-software tutorial. The speaker mixes Hindi and English (Hinglish); Hindi may be written in Devanagari and English in Latin script, and words of either language may appear in either script.

Write a summary in English of 120-180 words covering the main steps, concepts and facts taught. Write plain prose: no bullet points, no title.

Transcript:
<<<
{transcript}
>>>"""
USER_SUM_HING = USER_SUM_EN.replace(
    "Write a summary in English of 120-180 words",
    "Write a summary in Hinglish (Hindi-English mix written in Roman script, the way students commonly write) of 120-180 words")
assert USER_SUM_HING != USER_SUM_EN

TEMPLATES = dict(reference=(SYSTEM_REF, USER_REF), lexicon=(SYSTEM_LEX, USER_LEX), judge=(SYSTEM_JUDGE, USER_JUDGE),
                 judge_window=(SYSTEM_JUDGE, USER_JUDGE_BATCH),
                 summary_en=(SYSTEM_SUM, USER_SUM_EN), summary_hinglish=(SYSTEM_SUM, USER_SUM_HING))
for tid, (s, u) in TEMPLATES.items():
    log_prompt("2-templates", tid, GEMINI_MODEL if tid in ("reference", "lexicon", "judge", "judge_window") else QWEN_MODEL,
               dict(system=s, user_template=u), dict(template_sha=sha([s, u])))
update_run_config(templates={k: sha(v) for k, v in TEMPLATES.items()}, gemini_send_temperature=SEND_TEMPERATURE)

In [ ]:
# ===================== STAGE 2: SCRIPT-SHIFT LEXICON (Gemini) =====================
en_types = sorted({t for r in sel.ref_norm for t in r.split() if token_script(t) == "en"}) if RUN_SCRIPT_SHIFT else []
if not RUN_SCRIPT_SHIFT:
    print("RUN_SCRIPT_SHIFT = False -> lexicon stage skipped (no Gemini calls), script-shift condition not built")
LEX_BATCH = 150
batches = list(range(0, len(en_types), LEX_BATCH))
if en_types:
    print(f"English word types: {len(en_types)} -> {len(batches)} batch(es) of up to {LEX_BATCH}", flush=True)
lexicon, stopped = {}, None
t_stage = time.time()
for k, b in enumerate(batches, start=1):
    words = en_types[b:b + LEX_BATCH]
    print(f"[{k}/{len(batches)}] words {b + 1}-{b + len(words)} ...", flush=True)
    t0 = time.time()
    try:
        lx, _ = gemini_json("lexicon", SYSTEM_LEX, USER_LEX.format(words="\n".join(words)), Lexicon,
                            max_output_tokens=8192, meta=dict(batch=b // LEX_BATCH))
    except GeminiStop as e:
        stopped = str(e)
        print(f"[{k}/{len(batches)}] stopped: {e}", flush=True)
        break
    asked, added = set(words), 0
    for p in lx.items:
        w_ = norm_text(p.word); d_ = norm_text(p.devanagari)
        if w_ in asked and d_ and DEVA_RE.search(d_) and not LATIN_RE.search(d_) and len(d_.split()) <= 3:
            lexicon[w_] = d_.replace(" ", "")                    # keep token count identical
            added += 1
    left = (time.time() - t_stage) / k * (len(batches) - k) / 60
    print(f"[{k}/{len(batches)}] +{added}/{len(words)} entries in {time.time() - t0:.0f}s "
          f"| total {len(lexicon)}/{len(en_types)} | ~{left:.0f} min left", flush=True)
lex_df = pd.DataFrame(sorted(lexicon.items()), columns=["word", "devanagari"])
lex_df.to_csv(f"{S2_DIR}/script_shift_lexicon.csv", index=False)
if stopped:
    print("STOPPED:", stopped)
print(f"English word types: {len(en_types)} | lexicon entries: {len(lexicon)} ({len(lexicon) / max(1, len(en_types)):.1%})")
print("sample:", dict(list(lexicon.items())[:12]))

## Done?

- Last lines should say `lexicon entries: N (… %)` with no `STOPPED`. If it stopped on quota, run again later; it continues.
- Read 20 rows of `outputs/step2/script_shift_lexicon.csv` (e.g. click → क्लिक). Note how many look wrong.
- Then in the main notebook set `RUN_SCRIPT_SHIFT = True` and Run all: its lexicon stage should show no
  `calling Gemini` lines, and Qwen will make the 16 script-shift summaries (needs the T4).